In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import col, row_number, current_timestamp
from pyspark.sql.window import Window

# Đọc raw
df_raw_cust = spark.table("raw_customers")

# Làm sạch: loại bỏ null key, bỏ duplicate
df_clean_cust = df_raw_cust \
    .filter(col("customer_id").isNotNull()) \
    .dropDuplicates(["customer_id"])

# Tạo surrogate key
w = Window.orderBy("customer_id")
df_dim_customer = df_clean_cust \
    .withColumn("customer_key", row_number().over(w)) \
    .select(
        col("customer_key"),
        col("customer_id"),
        col("customer_name"),
        col("city"),
        col("segment")
    )

df_dim_customer.write.mode("overwrite").saveAsTable("DimCustomer")
print(f"DimCustomer: {df_dim_customer.count()} rows")
df_dim_customer.show()

StatementMeta(, 6836dc74-2984-41dd-aaae-b0e5f5791ed9, 3, Finished, Available, Finished, False)

DimCustomer: 5 rows
+------------+-----------+-------------+-------+---------+
|customer_key|customer_id|customer_name|   city|  segment|
+------------+-----------+-------------+-------+---------+
|           1|       C001| Nguyen Van A|  Hanoi|   Retail|
|           2|       C002|   Tran Thi B|   HCMC|Wholesale|
|           3|       C003|     Le Van C|Da Nang|   Retail|
|           4|       C004|   Pham Thi D|  Hanoi|   Online|
|           5|       C005|  Hoang Van E|   HCMC|   Retail|
+------------+-----------+-------------+-------+---------+



In [2]:
df_raw_prod = spark.table("raw_products")

df_clean_prod = df_raw_prod \
    .filter(col("product_id").isNotNull()) \
    .dropDuplicates(["product_id"])

w2 = Window.orderBy("product_id")
df_dim_product = df_clean_prod \
    .withColumn("product_key", row_number().over(w2)) \
    .select(
        col("product_key"),
        col("product_id"),
        col("product_name"),
        col("category"),
        col("unit_price").cast("double")
    )

df_dim_product.write.mode("overwrite").saveAsTable("DimProduct")
print(f"DimProduct: {df_dim_product.count()} rows")
df_dim_product.show()

StatementMeta(, 6836dc74-2984-41dd-aaae-b0e5f5791ed9, 5, Finished, Available, Finished, False)

DimProduct: 5 rows
+-----------+----------+-------------------+-----------+----------+
|product_key|product_id|       product_name|   category|unit_price|
+-----------+----------+-------------------+-----------+----------+
|          1|      P001|        Laptop Dell|Electronics|     1.5E7|
|          2|      P002|     Mouse Logitech|Electronics|  350000.0|
|          3|      P003|         Desk Chair|  Furniture| 2500000.0|
|          4|      P004|    Monitor Samsung|Electronics| 5000000.0|
|          5|      P005|Keyboard Mechanical|Electronics|  800000.0|
+-----------+----------+-------------------+-----------+----------+



In [3]:
from pyspark.sql.functions import to_date, monotonically_increasing_id

df_raw_sales = spark.table("raw_sales")
df_dim_cust = spark.table("DimCustomer")
df_dim_prod = spark.table("DimProduct")

# Làm sạch
df_clean_sales = df_raw_sales \
    .filter(col("order_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .filter(col("product_id").isNotNull()) \
    .filter(col("quantity").cast("int") > 0)

# Join để lấy surrogate keys
df_fact = df_clean_sales \
    .join(df_dim_cust.select("customer_key", "customer_id"), "customer_id", "left") \
    .join(df_dim_prod.select("product_key", "product_id"), "product_id", "left") \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("sales_key", monotonically_increasing_id()) \
    .select(
        col("sales_key"),
        col("order_id"),
        col("order_date"),
        col("customer_key"),
        col("product_key"),
        col("quantity").cast("int"),
        col("line_amount").cast("double").alias("revenue")
    )

df_fact.write.mode("overwrite").saveAsTable("FactSales")
print(f"FactSales: {df_fact.count()} rows")
df_fact.show()

StatementMeta(, 6836dc74-2984-41dd-aaae-b0e5f5791ed9, 7, Finished, Available, Finished, False)

FactSales: 8 rows
+---------+--------+----------+------------+-----------+--------+---------+
|sales_key|order_id|order_date|customer_key|product_key|quantity|  revenue|
+---------+--------+----------+------------+-----------+--------+---------+
|        0|  ORD001|2024-01-10|           1|          1|       1|    1.5E7|
|        1|  ORD001|2024-01-10|           1|          2|       2| 700000.0|
|        2|  ORD002|2024-01-11|           2|          3|       3|7500000.0|
|        3|  ORD003|2024-01-12|           3|          4|       1|5000000.0|
|        4|  ORD004|2024-01-13|           4|          1|       2|    3.0E7|
|        5|  ORD005|2024-01-14|           5|          5|       5|4000000.0|
|        6|  ORD006|2024-02-01|           1|          3|       1|2500000.0|
|        7|  ORD007|2024-02-05|           2|          4|       2|    1.0E7|
+---------+--------+----------+------------+-----------+--------+---------+



In [4]:
# Incremental load dựa vào last_modified

from pyspark.sql.functions import lit

LAST_LOAD_DATE = "2024-01-31"  # Thay đổi giá trị này mỗi lần chạy incremental

df_incr = spark.table("raw_sales") \
    .filter(col("last_modified") > LAST_LOAD_DATE)

if df_incr.count() > 0:
    df_incr_fact = df_incr \
        .join(df_dim_cust.select("customer_key", "customer_id"), "customer_id", "left") \
        .join(df_dim_prod.select("product_key", "product_id"), "product_id", "left") \
        .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
        .withColumn("sales_key", monotonically_increasing_id()) \
        .select("sales_key","order_id","order_date","customer_key","product_key",
                col("quantity").cast("int"), col("line_amount").cast("double").alias("revenue"))
    
    # Append (không overwrite) vào FactSales
    df_incr_fact.write.mode("append").saveAsTable("FactSales")
    print(f"Incremental: Appended {df_incr_fact.count()} new rows")
else:
    print("No new records to load")

StatementMeta(, 6836dc74-2984-41dd-aaae-b0e5f5791ed9, 9, Finished, Available, Finished, False)

Incremental: Appended 2 new rows
